<a href="https://colab.research.google.com/github/MarcinPlodzien/deep-learning-for-scientific-discovery/blob/main/notebooks/lecture-09/iris-diabetes-forests.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Random forests for classes and numbers

A random forest combines predictions from many randomized trees. Scikit-learn handles training internally. We fit a classifier to flower measurements and a regressor to diabetes measurements, then use their standard prediction methods.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import balanced_accuracy_score, root_mean_squared_error, ConfusionMatrixDisplay
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)
SEED = 1975
Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

In [2]:
iris = load_iris()
Xfit, Xtest, yfit, ytest = train_test_split(iris.data, iris.target, test_size=.2,
                                        stratify=iris.target, random_state=SEED)
classifier = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=2)
classifier.fit(Xfit, yfit)
print("Predicted species", classifier.predict(Xtest[:5]))
print("Probabilities", classifier.predict_proba(Xtest[:1]))
print("Test balanced accuracy", balanced_accuracy_score(ytest, classifier.predict(Xtest)))

Predicted species [0 1 2 0 0]
Probabilities [[1. 0. 0.]]
Test balanced accuracy 0.9333333333333332


The settings above were fixed in advance. A separate validation set is needed if settings are compared. Each probability vector sums to one, and the largest component determines the predicted class.

In [3]:
diabetes = load_diabetes(scaled=False)
Xfit, Xtest, yfit, ytest = train_test_split(diabetes.data[:, [2, 3]], diabetes.target,
                                        test_size=.2, random_state=SEED)
regressor = RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=SEED, n_jobs=2)
regressor.fit(Xfit, yfit)
print("Predicted scores", regressor.predict(Xtest[:5]))
print("Observed scores", ytest[:5])
print("Test RMSE", root_mean_squared_error(ytest, regressor.predict(Xtest)))

Predicted scores [112.33312541 127.86728199 195.12780838  95.14179157 131.40722796]
Observed scores [200. 258. 265.  74.  61.]
Test RMSE 70.17731563854807


Classification uses class labels and probabilities. Regression returns numerical responses. More trees increase computation and can stabilize the combined prediction, while a shared measurement problem can affect the whole forest. The model-comparison notebook adds validation and baselines for both tasks.